In [1]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
from datetime import datetime

In [2]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=Ventas_Comerssia;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [3]:
# ===============================
# Datos de clientes y ventas
# ===============================
df_clientes = pd.read_sql("SELECT Cliente, CosechaFecha FROM Ventas_Comerssia.dbo.Cliente_Perfil", engine)
df_clientes["CosechaFecha"] = pd.to_datetime(df_clientes["CosechaFecha"], errors="coerce")

In [5]:
df_excel = pd.read_excel("CONOCERTE.xlsx")

In [ ]:
df_nuevos

In [22]:
# ===============================
# 5) Función exacta para asignar segmento por monto (USANDO TUS UMbrales)
# ===============================
def segmento_por_valor(valor):
    # Nota: 0 => "Sin Segmento"
    if pd.isna(valor) or valor == 0:
        return "Sin Segmento"
    if valor > 1_400_000:
        return "Diamante"
    if 700_000 <= valor <= 1_400_000:
        return "Oro"
    if 300_000 <= valor < 700_000:
        return "Plata"
    if 0 < valor < 300_000:
        return "Bronce"
    return "Sin Segmento"

clientes["Segmento12M"] = clientes["Venta12M"].apply(segmento_por_valor)
clientes["Segmento24M"] = clientes["Venta24M"].apply(segmento_por_valor)

# ===============================
# 6) Recencia en días (sin huecos)
# ===============================
# calcular dias desde ultima compra (NaT -> NaN)
clientes["RecenciaDias"] = (fecha_corte - clientes["UltimaCompra"]).dt.days

def calcular_recencia(row):
    # Si no tiene ultima compra -> devolvemos None (mas tarde lo blankeamos si no tiene segmento)
    if pd.isna(row["UltimaCompra"]):
        return None

    # Nuevo: basado en cosecha (<= 90 dias)
    if pd.notna(row["CosechaFecha"]):
        diff_cosecha = (fecha_corte - row["CosechaFecha"]).days
        if diff_cosecha <= 90:
            return "Nuevo"

    d = row["RecenciaDias"]
    if d <= 120:
        return "Muy Activo"
    if 121 <= d <= 300:
        return "Activo"
    if 301 <= d <= 365:
        return "Por Inactivar"
    if 366 <= d <= 395:
        return "Churn"
    if d > 395:
        return "Inactivo"
    return None

clientes["Recencia"] = clientes.apply(calcular_recencia, axis=1)

# ===============================
# 7) Definir Segmento final (única columna) siguiendo tu regla
# ===============================
def asignar_segmento_final(row):
    rec = row["Recencia"]
    if rec in ["Nuevo", "Muy Activo", "Activo", "Por Inactivar"]:
        return row["Segmento12M"]
    if rec in ["Churn", "Inactivo"]:
        return row["Segmento24M"]
    # Si no tiene recencia (no compró) -> si tiene ventas24M==0 -> Sin Segmento
    return "Sin Segmento"

clientes["Segmento"] = clientes.apply(asignar_segmento_final, axis=1)

# ===============================
# 8) Regla final: si Segmento = "Sin Segmento" -> borrar Recencia (que quede vacío)
# ===============================
clientes.loc[clientes["Segmento"] == "Sin Segmento", "Recencia"] = None

In [23]:
# ===============================
# 9) Resultado final (columns)
# ===============================
resultado = clientes[[
    "Cliente", "CosechaFecha", "UltimaCompra",
    "Venta12M", "Venta24M",
    "Recencia", "Segmento"
]]

# Mostrar ejemplo
print(resultado.head(50))

        Cliente CosechaFecha UltimaCompra    Venta12M    Venta24M  \
0     C79354088   2025-02-15   2025-08-16   541680.66   541680.66   
1     C52307399   2020-11-20          NaT         NaN         NaN   
2     C52077631   2022-05-06   2025-02-22   930252.07   930252.07   
3   C1018442464   2020-02-02          NaT         NaN         NaN   
4     C66822518   2020-09-15          NaT         NaN         NaN   
5     C30388119   2021-06-21          NaT         NaN         NaN   
6      C1161079   2023-01-29          NaT         NaN         NaN   
7     C80412053   2019-11-14   2025-06-07   562605.05   675210.09   
8     C52217623   2021-03-08   2025-08-03  2550000.01  4022731.11   
9     C13862250   2022-10-14          NaT         NaN         NaN   
10    C53159445   2024-08-04   2024-08-04         NaN    78147.90   
11    C22493630   2024-12-22   2024-12-22   127731.09   127731.09   
12    C43483124   2024-03-20   2024-03-20         NaN   884873.95   
13  C1082931414   2023-11-14   202

In [26]:
resultado.to_excel("segmentacion_clientes2.xlsx", index=False)

In [27]:
resultado.to_sql("Segmentacion_Clientes", engine, if_exists="replace", index=False)

112